dataset.py



In [1]:
import os
import cv2
import numpy as np
from torch.utils.data import Dataset

class CellDataset(Dataset):
    def __init__(self, image_dir, mask_dir):
        self.image_dir = image_dir
        self.mask_dir = mask_dir
        self.images = os.listdir(image_dir)

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_path = os.path.join(self.image_dir, self.images[idx])
        mask_path = os.path.join(self.mask_dir, self.images[idx])

        image = cv2.imread(img_path)
        image = cv2.resize(image, (256, 256))
        image = image / 255.0

        mask = cv2.imread(mask_path, 0)
        mask = cv2.resize(mask, (256, 256))
        mask = np.expand_dims(mask, axis=0) / 255.0

        return image.transpose(2, 0, 1), mask

model.py (U-Net — minimum expected)

In [2]:
import torch
import torch.nn as nn

class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.conv(x)

class UNet(nn.Module):
    def __init__(self):
        super().__init__()

        self.down1 = DoubleConv(3, 64)
        self.down2 = DoubleConv(64, 128)
        self.pool = nn.MaxPool2d(2)

        self.bottleneck = DoubleConv(128, 256)

        self.up1 = nn.ConvTranspose2d(256, 128, 2, stride=2)
        self.conv1 = DoubleConv(256, 128)

        self.up2 = nn.ConvTranspose2d(128, 64, 2, stride=2)
        self.conv2 = DoubleConv(128, 64)

        self.final = nn.Conv2d(64, 1, 1)

    def forward(self, x):
        d1 = self.down1(x)
        d2 = self.down2(self.pool(d1))

        bottleneck = self.bottleneck(self.pool(d2))

        up1 = self.up1(bottleneck)
        up1 = torch.cat([up1, d2], dim=1)
        up1 = self.conv1(up1)

        up2 = self.up2(up1)
        up2 = torch.cat([up2, d1], dim=1)
        up2 = self.conv2(up2)

        return torch.sigmoid(self.final(up2))

train.py

In [7]:
import torch
from torch.utils.data import DataLoader
import os
import cv2
import numpy as np

import torch.nn as nn
import torch.optim as optim
from tqdm import tqdm

device = "cuda" if torch.cuda.is_available() else "cpu"

# Create the directories if they don't exist
os.makedirs("outputs/models", exist_ok=True)
os.makedirs("outputs/predictions", exist_ok=True)
os.makedirs("data/processed/images", exist_ok=True)
os.makedirs("data/processed/masks", exist_ok=True)

# Create a dummy image and mask if directories are empty
if not os.listdir("data/processed/images"):
    dummy_image_path = os.path.join("data/processed/images", "dummy_image.png")
    dummy_mask_path = os.path.join("data/processed/masks", "dummy_image.png")

    # Create a simple black image and mask (e.g., 256x256)
    dummy_image = np.zeros((256, 256, 3), dtype=np.uint8)
    dummy_mask = np.zeros((256, 256), dtype=np.uint8)

    cv2.imwrite(dummy_image_path, dummy_image)
    cv2.imwrite(dummy_mask_path, dummy_mask)

dataset = CellDataset("data/processed/images", "data/processed/masks")
loader = DataLoader(dataset, batch_size=8, shuffle=True)

model = UNet().to(device)
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

for epoch in range(10):
    loop = tqdm(loader)

    for images, masks in loop:
        images = images.float().to(device)
        masks = masks.float().to(device)

        preds = model(images)
        loss = criterion(preds, masks)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        loop.set_postfix(loss=loss.item())

torch.save(model.state_dict(), "outputs/models/unet.pth")

100%|██████████| 1/1 [00:02<00:00,  2.67s/it, loss=2.73e-17]


evaluate.py

In [10]:
import torch

import cv2
import numpy as np
import os

model = UNet()
model.load_state_dict(torch.load("outputs/models/unet.pth"))
model.eval()

# Create a dummy test.png if it doesn't exist
if not os.path.exists("test.png"):
    dummy_test_image = np.zeros((256, 256, 3), dtype=np.uint8)
    cv2.imwrite("test.png", dummy_test_image)

img = cv2.imread("test.png")
img = cv2.resize(img, (256, 256)) / 255.0
img = np.transpose(img, (2, 0, 1))
img = torch.tensor(img).unsqueeze(0).float()

with torch.no_grad():
    pred = model(img)

mask = pred.squeeze().numpy()
cv2.imwrite("outputs/predictions/mask.png", mask * 255)

True

app.py (UI Code — Clean & Working)

In [14]:
# To run this Streamlit app in Colab, you would typically save this code to a Python file (e.g., `app.py`) and then run it using `streamlit run app.py` in a terminal. Colab's interactive environment doesn't directly support running Streamlit apps within a cell in the same way a local terminal would.
# However, to address the 'ModuleNotFoundError', you first need to install the 'streamlit' library.
!pip install streamlit

import streamlit as st
import torch
import cv2
import numpy as np
# from src.model import UNet  <-- This import is not needed in Colab notebook environment

# Load model
@st.cache_resource
def load_model():
    model = UNet()
    model.load_state_dict(torch.load("outputs/models/unet.pth", map_location="cpu"))
    model.eval()
    return model

model = load_model()

st.title("🔬 Cell Segmentation Demo")
st.write("Upload a microscopy image to generate segmentation mask")

uploaded_file = st.file_uploader("Upload Image", type=["png", "jpg", "jpeg"])

if uploaded_file is not None:
    # Read image
    file_bytes = np.asarray(bytearray(uploaded_file.read()), dtype=np.uint8)
    image = cv2.imdecode(file_bytes, 1)

    st.subheader("Original Image")
    st.image(image, channels="BGR")

    # Preprocess
    img = cv2.resize(image, (256, 256)) / 255.0
    img = np.transpose(img, (2, 0, 1))
    img = torch.tensor(img).unsqueeze(0).float()

    # Prediction
    with torch.no_grad():
        pred = model(img)

    mask = pred.squeeze().numpy()
    mask = (mask > 0.5).astype(np.uint8) * 255

    st.subheader("Predicted Mask")
    st.image(mask, clamp=True)

    # Overlay
    overlay = cv2.addWeighted(image, 0.7, cv2.resize(mask, (image.shape[1], image.shape[0])), 0.3, 0)

    st.subheader("Overlay Result")
    st.image(overlay, channels="BGR")

2026-04-18 17:18:30.387 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-18 17:18:30.413 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-18 17:18:30.487 
  command:

    streamlit run /usr/local/lib/python3.12/dist-packages/colab_kernel_launcher.py [ARGUMENTS]
2026-04-18 17:18:30.487 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-18 17:18:30.489 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-18 17:18:30.491 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-18 17:18:30.493 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when runn

In [26]:
import subprocess
import time
import os

# --- Start of code to create app.py ---
# Define the content of the UNet model (from cell H4-cPW2mktO0)
unet_model_code = """
import torch
import torch.nn as nn

class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.conv(x)

class UNet(nn.Module):
    def __init__(self):
        super().__init__()

        self.down1 = DoubleConv(3, 64)
        self.down2 = DoubleConv(64, 128)
        self.pool = nn.MaxPool2d(2)

        self.bottleneck = DoubleConv(128, 256)

        self.up1 = nn.ConvTranspose2d(256, 128, 2, stride=2)
        self.conv1 = DoubleConv(256, 128)

        self.up2 = nn.ConvTranspose2d(128, 64, 2, stride=2)
        self.conv2 = DoubleConv(128, 64)

        self.final = nn.Conv2d(64, 1, 1)

    def forward(self, x):
        d1 = self.down1(x)
        d2 = self.down2(self.pool(d1))

        bottleneck = self.bottleneck(self.pool(d2))

        up1 = self.up1(bottleneck)
        up1 = torch.cat([up1, d2], dim=1)
        up1 = self.conv1(up1)

        up2 = self.up2(up1)
        up2 = torch.cat([up2, d1], dim=1)
        up2 = self.conv2(up2)

        return torch.sigmoid(self.final(up2))
"""

# Define the content of the Streamlit app logic (from cell HcpRNlqekzDd, excluding !pip install)
streamlit_app_logic = """
import streamlit as st
import torch
import cv2
import numpy as np

# Load model
@st.cache_resource
def load_model():
    model = UNet()
    model.load_state_dict(torch.load("outputs/models/unet.pth", map_location="cpu"))
    model.eval()
    return model

model = load_model()

st.title("🔬 Cell Segmentation Demo")
st.write("Upload a microscopy image to generate segmentation mask")

uploaded_file = st.file_uploader("Upload Image", type=["png", "jpg", "jpeg"])

if uploaded_file is not None:
    # Read image
    file_bytes = np.asarray(bytearray(uploaded_file.read()), dtype=np.uint8)
    image = cv2.imdecode(file_bytes, 1)

    st.subheader("Original Image")
    st.image(image, channels="BGR")

    # Preprocess
    img = cv2.resize(image, (256, 256)) / 255.0
    img = np.transpose(img, (2, 0, 1))
    img = torch.tensor(img).unsqueeze(0).float()

    # Prediction
    with torch.no_grad():
        pred = model(img)

    mask = pred.squeeze().numpy()
    mask = (mask > 0.5).astype(np.uint8) * 255

    st.subheader("Predicted Mask")
    st.image(mask, clamp=True)

    # Overlay
    # Resize mask to original image dimensions
    resized_mask = cv2.resize(mask, (image.shape[1], image.shape[0]))
    # Convert single-channel mask to 3-channel to match image
    resized_mask_colored = cv2.cvtColor(resized_mask, cv2.COLOR_GRAY2BGR)
    overlay = cv2.addWeighted(image, 0.7, resized_mask_colored, 0.3, 0)

    st.subheader("Overlay Result")
    st.image(overlay, channels="BGR")
"""

# Combine the UNet model definition and the Streamlit app logic into a single string
full_streamlit_app_content = unet_model_code + "\n" + streamlit_app_logic

# Write the combined content to app.py
with open("app.py", "w") as f:
    f.write(full_streamlit_app_content)
# --- End of code to create app.py ---

# Run the Streamlit app using subprocess to keep the kernel alive
# You might need to install 'ngrok' and set it up to expose the Streamlit app
!pip install pyngrok
from pyngrok import ngrok

# !!! IMPORTANT: Replace 'YOUR_AUTHTOKEN' with your actual ngrok authtoken !!!
# You can get your authtoken from https://dashboard.ngrok.com/get-started/your-authtoken
ngrok.set_auth_token("32YuyrAW3D1qJfvmqlSHzqjsWSd_BD96JnyTZMuhynBTTyvY")

# Kill any existing ngrok processes before starting a new one
!killall ngrok > /dev/null 2>&1

public_url = ngrok.connect(addr=8501)
print(f"Streamlit app available at: {public_url}")

# Terminate any previous Streamlit processes to avoid port conflicts
!killall streamlit > /dev/null 2>&1

process = subprocess.Popen(['streamlit', 'run', 'app.py'], stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)

print("Streamlit app started. Monitoring its output...")

# Read and print output from the Streamlit subprocess for a few seconds
start_time = time.time()
while time.time() - start_time < 10:
    stdout_line = process.stdout.readline()
    stderr_line = process.stderr.readline()
    if stdout_line:
        print(f"[Streamlit STDOUT] {stdout_line.strip()}")
    if stderr_line:
        print(f"[Streamlit STDERR] {stderr_line.strip()}")
    if not stdout_line and not stderr_line and process.poll() is not None:
        break # Process exited
    time.sleep(0.1)

# Check if the process is still running after initial monitoring
if process.poll() is None:
    print("Streamlit process appears to be running in the background.")
else:
    print(f"Streamlit process exited with code {process.returncode}")
    stdout, stderr = process.communicate()
    if stdout: print(f"[Final Streamlit STDOUT] {stdout.strip()}")
    if stderr: print(f"[Final Streamlit STDERR] {stderr.strip()}")

Streamlit app available at: NgrokTunnel: "https://ebca-136-107-237-52.ngrok-free.app" -> "http://localhost:8501"
Streamlit app started. Monitoring its output...


ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.



Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py", line 3553, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
  File "/tmp/ipykernel_11175/512646983.py", line 149, in <cell line: 0>
    stderr_line = process.stderr.readline()
                  ^^^^^^^^^^^^^^^^^^^^^^^^^
KeyboardInterrupt

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py", line 2099, in showtraceback
    stb = value._render_traceback_()
          ^^^^^^^^^^^^^^^^^^^^^^^^
AttributeError: 'KeyboardInterrupt' object has no attribute '_render_traceback_'

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/IPython/core/ultratb.py", line 1101, in get_records
    return _fixed_getinnerframes(etb, number_of_lines

TypeError: object of type 'NoneType' has no len()

In [24]:
!killall ngrok